
# ARGOX — End‑to‑End (FINAL)  
**Date:** 2025-11-12

A streamlined notebook that runs **end‑to‑end** using **existing local cache files** (no external pulls).  
It combines the robust loaders/plotting from **Fixed15** with early‑pipeline sanity checks from **Fixed10**.



## 0) Configuration
Leave entries as `None` to auto-detect; set a path to override.


In [1]:
# =========================
# 1) IMPORTS + CONFIG / PATHS
# =========================
# Run this cell FIRST.

import os, glob, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Output / cache dirs
OUT_DIR  = './outputs4'   # all new outputs written here
CACHE_DIR = './cache'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# -------------------------
# Inputs (edit to match your WD)
# -------------------------

# Weekly R(t) input (already weekly)
RT_CSV = './cache/rt_state_weekly.csv'

# Daily absolute humidity (optional; used only for the daily-Rt reconstruction section)
HUMIDITY_DAILY_CSV = './cache/ah_daily_allstates.csv'

# ARGO-adjusted weekly proxy (preferred for daily-Rt reconstruction)
# Expected columns: date, state, pred
ARGO_PROXY_LONG_CSV = None  # e.g., './out/argox/state_preds_argo_step2_long.csv'

# SafeGraph county-to-county input directory (weekly CSVs)
SAFEGRAPH_DIR = './mobility df'
SAFEGRAPH_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]  # include 2019 explicitly

# Mobility cache built from SafeGraph county-to-county files (written under CACHE_DIR)
MOBILITY_TOTAL_CSV  = os.path.join(CACHE_DIR, 'mobility_state_weekly_total_fromSafeGraph.csv')
FORCE_REBUILD_MOB = True

# Optional: ILI / GT caches (not required for this notebook)
ILI_CACHE_FILE = None
GT_CACHE_DIR = None

print('[info] Config loaded. OUT_DIR=', OUT_DIR, '| CACHE_DIR=', CACHE_DIR)


[info] Config loaded. OUT_DIR= ./outputs4 | CACHE_DIR= ./cache



## 1) Imports & Utilities


In [2]:
# =========================
# 2) Utilities
# =========================

def _find_first_existing(candidates, search_dirs=(".",)):
    """Return first path that exists given candidate filenames and search dirs."""
    for sd in search_dirs:
        for c in candidates:
            fp = os.path.join(sd, c)
            if os.path.exists(fp):
                return fp
    return None

def _preview(df, name, n=5):
    print(f'\n--- {name} (n={len(df)}) ---')
    display(df.head(n))



## 2) ILI/GT/Humidity sanity (optional)


In [3]:
# (Optional) ILI loader block removed for this Figure 1A notebook.
# This notebook operates on:
#   - weekly R(t) (RT_CSV)
#   - weekly mobility from SafeGraph (rebuilt or cached)
#   - optional daily AH + weekly ARGO proxy for reconstructed daily R(t)
#
# If you want to load ILI for other analyses, do it in a separate notebook/module.



## 3) Mobility + R(t) loaders (auto-detect with fallbacks)


In [5]:
# ============================
# 3A) Build weekly state mobility from SafeGraph (county-to-county)
#     - outputs ONE weekly series per state:
#         1) total weekly inflow visits (raw)
#
# Notes:
# - This pipeline is WEEKLY. It does NOT compute a true "day-of-week baseline % change" metric.
# - SAFEGRAPH_YEARS explicitly includes 2019 per your request.
# ============================

import os, glob
import numpy as np
import pandas as pd

# ---- Ensure FIPS_TO_STATE mapping exists (load from CSV if needed) ----
def _ensure_fips_to_state(fips_map_csv='./cache/config/state_fips_map.csv'):
    global FIPS_TO_STATE
    if 'FIPS_TO_STATE' in globals() and isinstance(FIPS_TO_STATE, dict) and len(FIPS_TO_STATE) > 0:
        return

    if not os.path.exists(fips_map_csv):
        raise FileNotFoundError(
            f"Missing FIPS map CSV: {fips_map_csv}\n"
            "Expected it under ./cache/config/state_fips_map.csv (or update the path)."
        )

    fips_df = pd.read_csv(fips_map_csv, dtype=str)
    fips_df.columns = [c.lower().strip() for c in fips_df.columns]

    # accept common column names
    if 'state_fips' in fips_df.columns:
        sf = fips_df['state_fips']
    elif 'fips' in fips_df.columns:
        sf = fips_df['fips']
    else:
        raise ValueError("state_fips_map.csv must contain 'state_fips' or 'fips' column (2-digit state FIPS).")
    sf = sf.astype(str).str.extract(r'(\d+)')[0].str.zfill(2)

    if 'state' in fips_df.columns:
        ab = fips_df['state']
    elif 'state_abbr' in fips_df.columns:
        ab = fips_df['state_abbr']
    else:
        raise ValueError("state_fips_map.csv must contain 'state' or 'state_abbr' column (state abbreviation).")
    ab = ab.astype(str).str.strip().str.upper()

    FIPS_TO_STATE = dict(zip(sf, ab))
    print(f"[info] Loaded FIPS_TO_STATE mapping from {fips_map_csv} ({len(FIPS_TO_STATE)} entries)")

_ensure_fips_to_state()

def _week_end_saturday(s):
    # Convert a datetime-like Series to week-ending Saturday stamps
    s = pd.to_datetime(s, errors='coerce')
    return s.dt.to_period('W-SAT').dt.end_time.dt.normalize()

def _detect_cols(cols_lower):
    # Return (date_col, dest_col, visits_col) using common SafeGraph naming variants.
    date_cands = [
        'week_start', 'week_end',
        'date_range_start', 'date_range_end',
        'start_date', 'end_date',
        'date', 'week',
    ]
    dest_cands = [
        'destination_fips', 'dest_fips', 'destination_county_fips', 'dest',
        'destination', 'dst_fips',
    ]
    visits_cands = [
        'total_visits', 'totalvisits', 'visits', 'visit_count',
        'raw_visit_counts', 'raw_visit_count',
        'total_visit_counts', 'total_visit_count',
        'visits_total',
    ]
    date_col = next((c for c in date_cands if c in cols_lower), None)
    dest_col = next((c for c in dest_cands if c in cols_lower), None)
    visits_col = next((c for c in visits_cands if c in cols_lower), None)
    return date_col, dest_col, visits_col

def build_state_weekly_mobility_from_safegraph(files, out_total_csv, force=False):
    if (not force) and os.path.exists(out_total_csv):
        print('[mob_cache] Using existing cached mobility CSV.')
        return

    rows = []
    for fp in files:
        # read only header first to detect columns robustly
        try:
            head = pd.read_csv(fp, nrows=5, low_memory=False)
        except Exception as e:
            print('[mob_cache][warn] head read failed:', os.path.basename(fp), e)
            continue

        cols_map = {c.lower().strip(): c for c in head.columns}
        cols_lower = set(cols_map.keys())
        date_l, dest_l, visits_l = _detect_cols(cols_lower)

        if date_l is None or dest_l is None or visits_l is None:
            sample_cols = list(head.columns)[:25]
            print('[mob_cache][warn] Could not detect required columns in', os.path.basename(fp))
            print('            detected date/dest/visits:', date_l, dest_l, visits_l)
            print('            first columns:', sample_cols)
            continue

        date_col = cols_map[date_l]
        dest_col = cols_map[dest_l]
        visits_col = cols_map[visits_l]

        try:
            df = pd.read_csv(fp, usecols=[date_col, dest_col, visits_col], low_memory=False)
        except Exception as e:
            print('[mob_cache][warn] read failed:', os.path.basename(fp), e)
            continue

        d = pd.DataFrame({
            'date': pd.to_datetime(df[date_col], errors='coerce'),
            'dest_fips': df[dest_col],
            'visits': pd.to_numeric(df[visits_col], errors='coerce')
        }).dropna(subset=['date', 'visits'])

        # normalize county fips to 5-digit string
        fips5 = d['dest_fips'].astype(str).str.extract(r'(\d+)')[0].str.zfill(5)
        state2 = fips5.str[:2]
        d['state'] = state2.map(FIPS_TO_STATE)
        d = d.dropna(subset=['state'])

        # stamp to week-ending Saturday
        d['date'] = _week_end_saturday(d['date'])

        # aggregate to state-week totals
        g = d.groupby(['state', 'date'], as_index=False)['visits'].sum()
        rows.append(g)

    if not rows:
        print('[mob_cache] No rows produced; check SAFEGRAPH_DIR, file patterns, and column detection output above.')
        return

    allw = pd.concat(rows, ignore_index=True)
    allw = allw.groupby(['state', 'date'], as_index=False)['visits'].sum().sort_values(['state', 'date'])

    out_total = allw.rename(columns={'visits': 'mob_total'})
    out_total.to_csv(out_total_csv, index=False)
    print('[mob_cache] wrote:', out_total_csv, 'rows=', len(out_total))

# ---- Find SafeGraph files (your filenames match these patterns) ----
files = []
for y in SAFEGRAPH_YEARS:
    files.extend(glob.glob(os.path.join(SAFEGRAPH_DIR, f"*{y}*.csv")))
files = sorted(set(files))

if not files:
    print('[mob_cache] No SafeGraph county-to-county files found under', SAFEGRAPH_DIR)
else:
    print(f"[mob_cache] Found {len(files)} SafeGraph files. Example: {os.path.basename(files[0])}")
    build_state_weekly_mobility_from_safegraph(
        files,
        out_total_csv=MOBILITY_TOTAL_CSV,
        force=FORCE_REBUILD_MOB
    )


[info] Loaded FIPS_TO_STATE mapping from ./cache/config/state_fips_map.csv (51 entries)
[mob_cache] Found 8 SafeGraph files. Example: visits_county_to_county_US_2019.csv
[mob_cache] wrote: ./cache/mobility_state_weekly_total_fromSafeGraph.csv rows= 15823


In [6]:
# ============================
# 3B) Load weekly mobility + weekly R(t) and ALIGN on identical weekly dates
#     - Produces aligned panel per state:
#         (A) R(t) vs weekly mobility total
# ============================

import os
import numpy as np
import pandas as pd

# ---- Explicit paths (your WD: /argo_seir_eakf) ----
RT_CSV = './cache/rt_state_weekly.csv'
MOBILITY_TOTAL_CSV  = './cache/mobility_state_weekly_total_fromSafeGraph.csv'

def _find_first_existing(candidates, search_dirs=('.', './cache','./outputs','./outputs2','./outputs3','./outputs4','./cache/ili_cache','./cache/configs')):
    for d in search_dirs:
        for c in candidates:
            p = os.path.join(d, c)
            if os.path.exists(p):
                return p
    return None

def _pick_date_col(df):
    for c in ['date','week','week_end','week_ending','week_start','date_range_start','Week']:
        if c in df.columns:
            return c
    for c in df.columns:
        s = pd.to_datetime(df[c], errors='coerce')
        if s.notna().mean() > 0.8:
            return c
    return None

def _week_end_saturday_series(s):
    s = pd.to_datetime(s, errors='coerce')
    return s.dt.to_period('W-SAT').dt.end_time.dt.normalize()

def _load_weekly_rt(path):
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"RT CSV not found: {path}")

    df = pd.read_csv(path)
    date_col = _pick_date_col(df)
    if date_col is None:
        raise ValueError(f'Could not find a date column in RT CSV. Columns={list(df.columns)[:25]}')

    state_col = next((c for c in ['state','abbr','state_abbr','location','loc'] if c in df.columns), None)
    if state_col is None:
        raise ValueError(f'Could not find a state column in RT CSV. Columns={list(df.columns)[:25]}')

    rt_col = next((c for c in ['rt','r_t','rt_weekly','rt_median','re','r_eff','r_effective'] if c in df.columns), None)
    if rt_col is None:
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        best, best_score = None, -1
        for c in num_cols:
            s = pd.to_numeric(df[c], errors='coerce')
            med = float(np.nanmedian(s))
            if np.isnan(med):
                continue
            score = 0
            if 0.5 <= med <= 2.0: score += 2
            if 0.1 <= med <= 4.0: score += 1
            if score > best_score:
                best_score, best = score, c
        rt_col = best

    if rt_col is None:
        raise ValueError(f'Could not identify an Rt column in RT CSV. Columns={list(df.columns)[:25]}')

    out = df[[state_col, date_col, rt_col]].copy()
    out.columns = ['state','date','rt']
    out['state'] = out['state'].astype(str).str.upper().str.strip()
    out['date']  = _week_end_saturday_series(out['date'])
    out['rt']    = pd.to_numeric(out['rt'], errors='coerce')
    out = out.dropna(subset=['state','date','rt'])
    out = out.groupby(['state','date'], as_index=False)['rt'].median()
    return out

def _load_weekly_mob_total(path):
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"Mobility CSV not found: {path}")

    df = pd.read_csv(path)
    date_col = _pick_date_col(df)
    if date_col is None:
        raise ValueError(f'Could not find a date column in {path}. Columns={list(df.columns)[:25]}')

    state_col = next((c for c in ['state','abbr','state_abbr','location','loc'] if c in df.columns), None)
    if state_col is None:
        raise ValueError(f'Could not find a state column in {path}. Columns={list(df.columns)[:25]}')

    val_col = next((c for c in ['mob_total','visits','totalVisits','totalvisits','mob'] if c in df.columns), None)
    if val_col is None:
        num_cols = [c for c in df.columns if c not in [state_col, date_col] and pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            raise ValueError(f'Could not identify a mobility value column in {path}. Columns={list(df.columns)[:25]}')
        val_col = num_cols[0]

    out = df[[state_col, date_col, val_col]].copy()
    out.columns = ['state','date','mob_total']
    out['state'] = out['state'].astype(str).str.upper().str.strip()
    out['date']  = _week_end_saturday_series(out['date'])
    out['mob_total'] = pd.to_numeric(out['mob_total'], errors='coerce')
    out = out.dropna(subset=['state','date','mob_total'])
    out = out.groupby(['state','date'], as_index=False)['mob_total'].sum()
    return out

# ---- If explicit paths missing, fall back to auto-detect ----
if not os.path.exists(RT_CSV):
    RT_CSV = _find_first_existing(['rt_state_weekly.csv','Rt_weekly_long.csv','Rt_weekly_wide.csv'])
if not os.path.exists(MOBILITY_TOTAL_CSV):
    MOBILITY_TOTAL_CSV = _find_first_existing(['mobility_state_weekly_total_fromSafeGraph.csv','mobility_state_weekly_total.csv'])

print('[info] Using RT_CSV:         ', RT_CSV)
print('[info] Using mobility TOTAL: ', MOBILITY_TOTAL_CSV)

# ---- Load + Align ----
rt_weekly = _load_weekly_rt(RT_CSV)
mob_total = _load_weekly_mob_total(MOBILITY_TOTAL_CSV)

tmp_total = rt_weekly.merge(mob_total, on=['state','date'], how='inner')

print('[align] Matched rows (TOTAL): ', len(tmp_total))

mcount = tmp_total.groupby('state').size()
rcount = rt_weekly.groupby('state').size()
frac = (mcount / rcount).sort_values()
print('[align] States with lowest match fraction (TOTAL):')
print(frac.head(10))

df_weekly_total = tmp_total


[info] Using RT_CSV:          ./cache/rt_state_weekly.csv
[info] Using mobility TOTAL:  ./cache/mobility_state_weekly_total_fromSafeGraph.csv
[align] Matched rows (TOTAL):  15505
[align] States with lowest match fraction (TOTAL):
state
VT    0.650964
DE    0.661670
MD    0.665953
RI    0.665953
MA    0.665953
CT    0.665953
PA    0.665953
HI    0.665953
NJ    0.665953
NH    0.668094
dtype: float64



## 4) EAKF/SIR hooks (optional)


In [7]:

EAKF_STATE_TRAJ = None
PARAM_PRIORS    = None

eakf_df=None
if EAKF_STATE_TRAJ and os.path.exists(EAKF_STATE_TRAJ):
    try:
        eakf_df=pd.read_csv(EAKF_STATE_TRAJ, parse_dates=['date'])
        preview(eakf_df,'EAKF state trajectory')
    except Exception as e:
        print('[warn] EAKF load failed:', e)
else:
    print('[info] No EAKF trajectory provided (optional).')

priors_df=None
if PARAM_PRIORS and os.path.exists(PARAM_PRIORS):
    try:
        priors_df=pd.read_csv(PARAM_PRIORS)
        preview(priors_df,'Parameter priors (Shaman/Yang/Lipsitch)')
    except Exception as e:
        print('[warn] Priors load failed:', e)


[info] No EAKF trajectory provided (optional).



## 5) Figures


In [8]:
# ============================
# Figure 1A/1C utilities (Spearman by period + plotting)
# ============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def _spearman(x, y):
    """Spearman rho + p-value (SciPy if available; rank-corr fallback otherwise)."""
    try:
        from scipy.stats import spearmanr
        rho, p = spearmanr(x, y, nan_policy='omit')
        return float(rho), float(p)
    except Exception:
        xr = pd.Series(x).rank()
        yr = pd.Series(y).rank()
        r = np.corrcoef(xr, yr)[0, 1]
        return float(r), np.nan

# --- Date-based period cutoffs requested ---
CUTOFF_1 = pd.Timestamp("2020-03-01")
CUTOFF_2 = pd.Timestamp("2022-09-01")

PERIOD_BINS = [
    (f"pre {CUTOFF_1.date()}", None, CUTOFF_1),
    (f"{CUTOFF_1.date()}–{(CUTOFF_2 - pd.Timedelta(days=1)).date()}", CUTOFF_1, CUTOFF_2),
    (f"post {CUTOFF_2.date()}", CUTOFF_2, None),
]

def compute_period_spearmans(df, rt_col="rt", mob_col="mob", min_n=3):
    """
    Computes:
      - overall Spearman over ALL rows in df (after dropna)
      - Spearman within 3 DATE windows:
           1) date < 2020-03-01
           2) 2020-03-01 <= date < 2022-09-01
           3) date >= 2022-09-01
    Assumes df is already restricted to flu-season weeks (your upstream pipeline does this).
    """
    if "date" in df.columns:
        d = df.copy()
        d["date"] = pd.to_datetime(d["date"])
        d = d.set_index("date")
    else:
        d = df.copy()
        d.index = pd.to_datetime(d.index)

    d = d[[rt_col, mob_col]].dropna()
    if d.empty:
        return {"overall": (np.nan, np.nan), "periods": {lab: (np.nan, np.nan) for lab,_,_ in PERIOD_BINS}}

    out = {"periods": {}, "n_periods": {}}

    # overall pooled
    out["overall"] = _spearman(d[rt_col], d[mob_col]) if len(d) >= min_n else (np.nan, np.nan)
    out["n_overall"] = int(len(d))

    # period windows
    for lab, start, end in PERIOD_BINS:
        dd = d.copy()
        if start is not None:
            dd = dd[dd.index >= start]
        if end is not None:
            dd = dd[dd.index < end]
        out["periods"][lab] = _spearman(dd[rt_col], dd[mob_col]) if len(dd) >= min_n else (np.nan, np.nan)
        out["n_periods"][lab] = int(len(dd))

    return out

def plot_rt_vs_mobility_fig1A(state, rt_series, mob_series, savepath, title_extra='', stats=None):
    """
    Plot R(t) and mobility on twin axes (smoothed for display).
    Title includes:
      - Overall Spearman rho/p over all available (flu-season) weeks in the series
      - Period Spearmans for the 3 date windows
    """
    df = pd.DataFrame({'rt': rt_series, 'mob': mob_series}).dropna()
    if df.empty:
        return

    if stats is None:
        stats = compute_period_spearmans(df, rt_col="rt", mob_col="mob")

    rho, pval = stats.get("overall", (np.nan, np.nan))

    # --- Smoothed series for plotting only ---
    sm = df.rolling(3, center=True, min_periods=1).mean()

    # --- Plot ---
    fig, ax1 = plt.subplots(figsize=(9.5, 4.8))
    ax2 = ax1.twinx()

    ax1.plot(sm.index, sm['rt'], color='tab:blue', lw=2.6, label='R(t)')
    ax2.plot(sm.index, sm['mob'], color='tab:orange', lw=2.2, alpha=0.9,
             label=title_extra if title_extra else 'Mobility')

    ax1.axhline(1.0, color='0.5', ls='--', lw=1)
    ax1.axhline(0.9, color='0.7', ls='--', lw=0.8)

    ax1.set_ylabel('R(t)', color='tab:blue')
    ax2.set_ylabel('Mobility (total visits)', color='tab:orange')
    ax1.tick_params(axis='y', colors='tab:blue')
    ax2.tick_params(axis='y', colors='tab:orange')

    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    for t in ax1.get_xticklabels():
        t.set_rotation(45)

    # --- Title text ---
    lines = []
    lines.append(f"{state} | Overall ρ={rho:.3f}, p={pval:.2g} (n={stats.get('n_overall',0)})")
    for lab, _, _ in PERIOD_BINS:
        rr, pp = stats["periods"].get(lab, (np.nan, np.nan))
        nn = stats.get("n_periods", {}).get(lab, 0)
        lines.append(f"{lab}: ρ={rr:.3f}, p={pp:.2g} (n={nn})")
    if title_extra:
        lines.append(title_extra)

    ax1.set_title("\n".join(lines), fontsize=10)

    # legend
    ax1.legend(loc='upper left', frameon=False)
    ax2.legend(loc='upper right', frameon=False)

    fig.tight_layout()
    fig.savefig(savepath, dpi=200)
    plt.close(fig)

def plot_fig1C_grid(stats_df, period_labels, out_png, out_csv, title):
    """
    Grid figure: rows=states, cols=periods; cell shows rho and p.
    Bold/border cells with p<0.05.
    """
    import matplotlib.pyplot as plt

    # Write CSV
    stats_df.to_csv(out_csv, index=False)

    states = stats_df["state"].tolist()
    nrow = len(states)
    ncol = len(period_labels)

    fig_h = max(10, 0.25 * nrow)
    fig_w = max(8, 2.6 * ncol)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.set_axis_off()

    # Build cell text matrix
    cell_text = []
    signif = []
    for _, r in stats_df.iterrows():
        row_txt = []
        row_sig = []
        for lab in period_labels:
            rho = r.get(f"{lab}_rho", np.nan)
            p = r.get(f"{lab}_p", np.nan)
            if pd.isna(rho) or pd.isna(p):
                row_txt.append("NA")
                row_sig.append(False)
            else:
                row_txt.append(f"{rho:.2f}\n(p={p:.2g})")
                row_sig.append(bool(p < 0.05))
        cell_text.append(row_txt)
        signif.append(row_sig)

    table = ax.table(
        cellText=cell_text,
        rowLabels=states,
        colLabels=period_labels,
        loc='center',
        cellLoc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.0, 1.2)

    # Emphasize significant cells
    for i in range(nrow):
        for j in range(ncol):
            cell = table[(i+1, j)]  # +1 because row 0 is header
            if signif[i][j]:
                cell.set_text_props(fontweight='bold')
                cell.set_linewidth(2.0)
            else:
                cell.set_linewidth(0.5)

    ax.set_title(title, fontsize=12, pad=20)
    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.close(fig)


In [9]:
# ============================
# Figure 1A (weekly) + Figure 1C (period grid)
#   - Uses weekly R(t) aligned to weekly SafeGraph mobility (TOTAL only)
#   - Computes Spearman within 3 DATE windows and overall
#   - Saves to outputs4 (overrides prior runs)
# ============================

import os
import pandas as pd

OUT_DIR_FIGS = './outputs4'
os.makedirs(OUT_DIR_FIGS, exist_ok=True)

states = sorted(df_weekly_total['state'].unique())

rows_total = []
count_total = 0

for st in states:
    d1 = df_weekly_total[df_weekly_total['state'] == st].set_index('date').sort_index()
    if d1[['rt','mob_total']].dropna().empty:
        continue

    stats_total = compute_period_spearmans(d1.rename(columns={'mob_total':'mob'}), rt_col='rt', mob_col='mob')

    out_png = os.path.join(OUT_DIR_FIGS, f'fig1A_{st}_rt_vs_mobTotal_weekly.png')
    plot_rt_vs_mobility_fig1A(
        st,
        d1['rt'],
        d1['mob_total'],
        out_png,
        title_extra='Weekly mobility (total visits)',
        stats=stats_total
    )
    count_total += 1

    row = {"state": st}
    for per, (r, p) in stats_total["periods"].items():
        row[f"{per}_rho"] = r
        row[f"{per}_p"] = p
    row["overall_rho"] = stats_total["overall"][0]
    row["overall_p"] = stats_total["overall"][1]
    rows_total.append(row)

print(f'[done] Figure 1A weekly plots saved to {OUT_DIR_FIGS}: total={count_total}')
print('[note] Mobility per-capita outputs removed (Spearman within-state is invariant to constant scaling).')

# ---- Figure 1C grid (TOTAL) ----
stats_total_df = pd.DataFrame(rows_total).sort_values("state").reset_index(drop=True)

period_labels = [lab for lab,_,_ in PERIOD_BINS]
out_png = os.path.join(OUT_DIR_FIGS, 'fig1C_spearman_grid_weekly_mobTotal.png')
out_csv = os.path.join(OUT_DIR_FIGS, 'fig1C_spearman_grid_weekly_mobTotal.csv')

plot_fig1C_grid(
    stats_total_df,
    period_labels=period_labels,
    out_png=out_png,
    out_csv=out_csv,
    title='Figure 1C: Spearman(weekly R(t), weekly mobility total) by time window (p<0.05 bold/bordered)'
)

print('[done] Figure 1C grid saved:', out_png)
print('[done] Figure 1C table saved:', out_csv)


[done] Figure 1A weekly plots saved to ./outputs4: total=49
[note] Mobility per-capita outputs removed (Spearman within-state is invariant to constant scaling).
[done] Figure 1C grid saved: ./outputs4/fig1C_spearman_grid_weekly_mobTotal.png
[done] Figure 1C table saved: ./outputs4/fig1C_spearman_grid_weekly_mobTotal.csv



## 6) Run summary


In [10]:
print('[info] Figure 1B disabled (no-op cell). Figures 1A and 1C are saved to outputs4.')


[info] Figure 1B disabled (no-op cell). Figures 1A and 1C are saved to outputs4.


In [11]:
from pathlib import Path
pngs=sorted(Path('./outputs4').glob('*.png'))
print('PNG outputs in ./outputs4:')
for p in pngs:
    print(' -', p.name)


PNG outputs in ./outputs4:
 - fig1A_AK_rt_vs_mobTotal_weekly.png
 - fig1A_AL_rt_vs_mobTotal_weekly.png
 - fig1A_AR_rt_vs_mobTotal_weekly.png
 - fig1A_AZ_rt_vs_mobTotal_weekly.png
 - fig1A_CA_rt_vs_mobTotal_weekly.png
 - fig1A_CO_rt_vs_mobTotal_weekly.png
 - fig1A_CT_rt_vs_mobTotal_weekly.png
 - fig1A_DE_rt_vs_mobTotal_weekly.png
 - fig1A_GA_rt_vs_mobTotal_weekly.png
 - fig1A_HI_rt_vs_mobTotal_weekly.png
 - fig1A_IA_rt_vs_mobTotal_weekly.png
 - fig1A_ID_rt_vs_mobTotal_weekly.png
 - fig1A_IL_rt_vs_mobTotal_weekly.png
 - fig1A_IN_rt_vs_mobTotal_weekly.png
 - fig1A_KS_rt_vs_mobTotal_weekly.png
 - fig1A_KY_rt_vs_mobTotal_weekly.png
 - fig1A_LA_rt_vs_mobTotal_weekly.png
 - fig1A_MA_rt_vs_mobTotal_weekly.png
 - fig1A_MD_rt_vs_mobTotal_weekly.png
 - fig1A_ME_rt_vs_mobTotal_weekly.png
 - fig1A_MI_rt_vs_mobTotal_weekly.png
 - fig1A_MN_rt_vs_mobTotal_weekly.png
 - fig1A_MO_rt_vs_mobTotal_weekly.png
 - fig1A_MS_rt_vs_mobTotal_weekly.png
 - fig1A_MT_rt_vs_mobTotal_weekly.png
 - fig1A_NC_rt_vs_mobTo